## GenAI Disclosure Statement

Generative AI tools were used as learning aids. All analysis and conclusions are the team's own work.

---

# Notebook 05 — Robustness, Drift, and Monitoring Plan
### DNSC 6330 Responsible Machine Learning Capstone | GWU

**Purpose:** Stress-test the model's stability under input distribution shift (PSI), calibration,
perturbation, and geographic holdout. Produce an operational monitoring playbook.

**Inputs:** `data/processed/*.parquet`, trained GBM model  
**Outputs:** `tables/psi_table.csv`, `tables/perturbation_table.csv`,
`tables/review_trigger_table.csv`, `figures/psi_*.png`, `figures/calibration_overall.png`

**Lecture 04 connection:** PSI, calibration, drift, monitoring triggers, subgroup monitoring.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
import warnings
warnings.filterwarnings("ignore")
import joblib
import glob

from src.robustness import psi_table, perturbation_test, calibration_stats, calibration_by_subgroup
from src.monitoring import get_playbook_table, evaluate_triggers

SEED = 42
np.random.seed(SEED)

PROC_DIR   = os.path.join(os.getcwd(), "..", "data", "processed")
MODELS_DIR = os.path.join(os.getcwd(), "..", "models")
TABLES_DIR = os.path.join(os.getcwd(), "..", "tables")
FIG_DIR    = os.path.join(os.getcwd(), "..", "figures")


In [ ]:
# Load all splits
train_df   = pd.read_parquet(os.path.join(PROC_DIR, "train.parquet"))
val_df     = pd.read_parquet(os.path.join(PROC_DIR, "val.parquet"))
test_df    = pd.read_parquet(os.path.join(PROC_DIR, "test.parquet"))
geo_df     = pd.read_parquet(os.path.join(PROC_DIR, "geo_holdout.parquet"))

NON_FEATURE_COLS = [
    "y", "action_taken", "state_code",
    "derived_race", "derived_sex", "derived_ethnicity", "race_sex_intersection"
]
feature_cols = [c for c in train_df.columns if c not in NON_FEATURE_COLS]

X_train = train_df[feature_cols]; y_train = train_df["y"].values
X_test  = test_df[feature_cols];  y_test  = test_df["y"].values
X_geo   = geo_df[feature_cols];   y_geo   = geo_df["y"].values

gbm_model = joblib.load(sorted(glob.glob(os.path.join(MODELS_DIR, "gbm_v*.pkl")))[-1])
metrics_df = pd.read_csv(os.path.join(TABLES_DIR, "metrics_table_final.csv"))
gbm_thresh_row = metrics_df[metrics_df["model"].str.contains("GBM")]
GBM_THRESHOLD = float(gbm_thresh_row["threshold"].values[0]) if len(gbm_thresh_row) > 0 else 0.5
print(f"Operating threshold: {GBM_THRESHOLD}")


## Section 5.1 — Population Stability Index (PSI)

PSI measures how much the distribution of a feature has shifted between the training
distribution and the geographic holdout (California), which simulates prospective drift.

**Bands:** PSI < 0.10: Stable | 0.10–0.25: Moderate | > 0.25: Major shift


In [ ]:
PSI_FEATURES = ["loan_amount", "income", "dti_numeric", "property_value",
                "combined_loan_to_value_ratio", "tract_minority_population_percent",
                "ffiec_msa_md_median_family_income", "tract_to_msa_income_percentage"]
PSI_FEATURES = [f for f in PSI_FEATURES if f in train_df.columns]

psi_df = psi_table(train_df, geo_df, PSI_FEATURES)
print("PSI Table — Training Distribution vs. Geographic Holdout (CA):")
display(psi_df)
psi_df.to_csv(os.path.join(TABLES_DIR, "psi_table.csv"), index=False)

# PSI bar chart
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#d32f2f" if v > 0.25 else ("#f57c00" if v > 0.10 else "#388e3c")
          for v in psi_df["psi"].fillna(0)]
bars = ax.barh(psi_df["feature"], psi_df["psi"].fillna(0), color=colors)
ax.axvline(x=0.10, color="orange", linestyle="--", lw=1.5, label="Moderate threshold (0.10)")
ax.axvline(x=0.25, color="red",    linestyle="--", lw=1.5, label="Major threshold (0.25)")
ax.set_xlabel("PSI")
ax.set_title("Population Stability Index\nTraining vs. Geographic Holdout (CA)", fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
psi_path = os.path.join(FIG_DIR, "psi_features.png")
plt.savefig(psi_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {psi_path}")


## Section 5.2 — Overall Calibration

Global calibration curve and Brier score on the test set.


In [ ]:
from sklearn.calibration import calibration_curve

test_probs = gbm_model.predict_proba(X_test)[:, 1]
cal_stats = calibration_stats(y_test, test_probs)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot([0, 1], [0, 1], "k--", lw=1.5, label="Perfect calibration")
ax.plot(cal_stats["mean_pred"], cal_stats["frac_pos"],
        "s-", color="steelblue", lw=2, markersize=6,
        label=f"GBM (ECE={cal_stats['ece']:.4f}, Brier={cal_stats['brier']:.4f})")
ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Fraction of Positives")
ax.set_title("Overall Calibration Curve\n(GBM, Test Set)", fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
cal_overall_path = os.path.join(FIG_DIR, "calibration_overall.png")
plt.savefig(cal_overall_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {cal_overall_path}")
print(f"Overall ECE: {cal_stats['ece']:.4f}")
print(f"Overall Brier: {cal_stats['brier']:.4f}")


## Section 5.3 — Geographic Holdout Stress Test

In [ ]:
from sklearn.metrics import roc_auc_score

geo_probs = gbm_model.predict_proba(X_geo)[:, 1]
geo_preds = (geo_probs >= GBM_THRESHOLD).astype(int)
geo_auc = roc_auc_score(y_geo, geo_probs)

test_auc = roc_auc_score(y_test, gbm_model.predict_proba(X_test)[:, 1])

print(f"Geographic Holdout Performance (CA):")
print(f"  Test AUC:        {test_auc:.4f}")
print(f"  Geo Holdout AUC: {geo_auc:.4f}  (Δ = {geo_auc - test_auc:+.4f})")
print(f"  Approval rate (test):     {(gbm_model.predict_proba(X_test)[:, 1] >= GBM_THRESHOLD).mean():.4f}")
print(f"  Approval rate (holdout):  {geo_preds.mean():.4f}")

# Fairness on geo holdout
if "derived_race" in geo_df.columns:
    from src.fairness import build_fairness_table
    geo_df_eval = geo_df.copy()
    geo_df_eval["y_prob"] = geo_probs
    geo_ft = build_fairness_table(
        geo_df_eval, y_true_col="y", y_prob_col="y_prob", threshold=GBM_THRESHOLD
    )
    geo_race = geo_ft[geo_ft["attribute"] == "Race"][["group", "approval_rate", "air"]]
    print("\nFairness on geographic holdout (Race AIR):")
    display(geo_race)


## Section 5.4 — Perturbation Testing

In [ ]:
PERTURB_FEATURES = [f for f in ["loan_amount", "income", "dti_numeric"]
                    if f in X_test.columns]

print(f"Running perturbation tests on: {PERTURB_FEATURES}")
pert_df = perturbation_test(
    gbm_model, X_test, y_test,
    features_to_perturb=PERTURB_FEATURES,
    noise_levels=(0.10, 0.20),
    fairness_df=test_df,
    protected_col="derived_race",
    reference_group="White",
)
print("Perturbation Test Results:")
display(pert_df)
pert_df.to_csv(os.path.join(TABLES_DIR, "perturbation_table.csv"), index=False)


## Section 5.5 — Monitoring and Review-Trigger Playbook

In [ ]:
playbook_df = get_playbook_table()
print("Monitoring Playbook (Review-Trigger Table):")
display(playbook_df)
playbook_df.to_csv(os.path.join(TABLES_DIR, "review_trigger_table.csv"), index=False)
print("\nPlaybook saved.")

# Evaluate current metrics against triggers
current_metrics = {
    "AUC (overall)": round(test_auc, 4),
}
# Add AIR from master fairness table if available
try:
    ft = pd.read_csv(os.path.join(TABLES_DIR, "master_fairness_table.csv"))
    for group, metric_name in [
        ("Black or African American", "AIR — Black applicants"),
        ("Hispanic or Latino", "AIR — Hispanic or Latino applicants"),
        ("Female", "AIR — Female applicants"),
    ]:
        row = ft[ft["group"] == group]
        if len(row) > 0 and pd.notna(row["air"].values[0]):
            current_metrics[metric_name] = float(row["air"].values[0])
    # Add PSI
    psi_loaded = pd.read_csv(os.path.join(TABLES_DIR, "psi_table.csv"))
    for feat, metric_name in [("loan_amount", "PSI — loan_amount"), ("income", "PSI — income")]:
        row = psi_loaded[psi_loaded["feature"] == feat]
        if len(row) > 0:
            current_metrics[metric_name] = float(row["psi"].values[0])
except Exception as e:
    print(f"Note: Could not load all metrics for trigger evaluation: {e}")

print("\nCurrent Metric Values vs. Review Triggers:")
trigger_status = evaluate_triggers(current_metrics)
display(trigger_status)
trigger_status.to_csv(os.path.join(TABLES_DIR, "trigger_status.csv"), index=False)
print("\n Notebook 05 complete.")
